# 3D GAT 分子打分器

## 单一模型：3D-enhanced GAT 预测全部 8 项性质

```
候选 SMILES -> RDKit 建图 (SMILES bonds)
            -> ETKDG+MMFF 生成 3D 坐标
            -> 计算 RBF 距离编码 (16 centers, 1.0-5.0 A)
            -> 3D GAT 预测全部 8 项性质
```

- 节点: 11 维 (SMILES 8d + BO 统计 3d, BO=0 at inference)
- 边: 23 维 (SMILES bonds 6d + RBF 16d + BO 1d, BO=0 at inference)
- 模型: 12-head GAT x 384d x 5 layers + metal-centric pooling


In [ ]:
# ============================================================
# Step 1: 导入 + 特征工程
# ============================================================
import numpy as np, pandas as pd, torch, torch.nn as nn, warnings, os
warnings.filterwarnings('ignore')
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger; RDLogger.logger().setLevel(RDLogger.ERROR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

ELEMENT_DATA = {
    1:(1,1),5:(2,13),6:(2,14),7:(2,15),8:(2,16),9:(2,17),
    14:(3,14),15:(3,15),16:(3,16),17:(3,17),
    20:(4,2),21:(4,3),22:(4,4),23:(4,5),24:(4,6),
    25:(4,7),26:(4,8),27:(4,9),28:(4,10),29:(4,11),30:(4,12),
    35:(4,17),42:(5,6),44:(5,8),45:(5,9),46:(5,10),
    47:(5,11),48:(5,12),53:(5,17),
    72:(6,4),73:(6,5),74:(6,6),75:(6,7),76:(6,8),
    77:(6,9),78:(6,10),79:(6,11),80:(6,12),
}
BOND_TYPES = [1, 2, 3, 12]
B2I = {b: i for i, b in enumerate(BOND_TYPES)}
N_BT = len(BOND_TYPES)

def atom_features(atom):
    z = atom.GetAtomicNum()
    p, g = ELEMENT_DATA.get(z, (0, 0))
    return [z, atom.GetDegree(), atom.GetFormalCharge(), p, g,
            int(atom.GetIsAromatic()), atom.GetTotalNumHs(), int(z >= 21)]

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float32)
    N = x.size(0)
    src, dst, battr = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = int(bond.GetBondType())
        src.extend([i, j]); dst.extend([j, i])
        bf = [0.0] * N_BT
        if bt in B2I: bf[B2I[bt]] = 1.0
        battr.extend([bf, bf])
    ei = torch.tensor([src, dst], dtype=torch.long) if src else torch.zeros(2,0,dtype=torch.long)
    ea = torch.tensor(battr, dtype=torch.float32) if battr else torch.zeros(0, N_BT)
    if ea.size(1) == N_BT:
        ea = torch.cat([ea, torch.zeros(ea.size(0), 1)], dim=1)
    return {'x': x, 'edge_index': ei, 'edge_attr': ea}


In [ ]:
# ============================================================
# Step 2: 3D GAT 单一模型 (SMILES GAT 已移除, 3D GAT 统一预测)
# ============================================================
# 所有 8 项性质均由 3D GAT 预测
# 坐标由 ETKDG+MMFF 自动生成, 无需外部文件
# Mayer BO 在推理时设为 0


In [ ]:
# ============================================================
# Step 3: 3D GAT 模型 + SMILES->3D图构建 (ETKDG+MMFF, BO=0)
# ============================================================

import os
CKPT_3D = '../chem_ml5/xyz_best_weighted.pt'
if not os.path.exists(CKPT_3D):
    CKPT_3D = '../chem_ml5/xyz_best.pt'  # fallback

import torch.nn.functional as F

HIDDEN=384; N_LAYERS=5; GAT_HEADS=12; DROPOUT=0.10

def segment_softmax(scores, dst, N):
    scores_f32 = scores.float()
    max_v = torch.full((N, scores_f32.size(1)), float('-inf'), device=scores_f32.device, dtype=torch.float32)
    max_v.scatter_reduce_(0, dst.unsqueeze(-1).expand(-1, scores_f32.size(1)), scores_f32, reduce='amax', include_self=False)
    exp_s = torch.exp(scores_f32 - max_v[dst])
    sum_v = torch.zeros(N, scores_f32.size(1), device=scores_f32.device, dtype=torch.float32)
    sum_v.scatter_add_(0, dst.unsqueeze(-1).expand(-1, scores_f32.size(1)), exp_s)
    return (exp_s / (sum_v[dst] + 1e-8)).to(scores.dtype)

class GATConv(nn.Module):
    def __init__(self, in_dim, out_dim_per_head, edge_dim=0, heads=4):
        super().__init__(); self.heads, self.out_dim = heads, out_dim_per_head
        total_dim = out_dim_per_head * heads
        self.node_lin = nn.Linear(in_dim, total_dim, bias=False)
        self.attn_src = nn.Linear(total_dim, heads, bias=False)
        self.attn_dst = nn.Linear(total_dim, heads, bias=False)
        self.edge_attn = nn.Linear(edge_dim, heads, bias=False) if edge_dim>0 else None
        self.act = nn.ReLU()
    def forward(self, x, ei, ea=None):
        src, dst = ei[0], ei[1]; N = x.size(0); h = self.node_lin(x)
        attn = self.attn_src(h[src]) + self.attn_dst(h[dst])
        if self.edge_attn is not None and ea is not None and ea.size(0)>0: attn = attn + self.edge_attn(ea)
        attn = F.leaky_relu(attn, 0.2)
        alpha = segment_softmax(attn, dst, N)
        h_heads = h.view(-1, self.heads, self.out_dim)
        msg = h_heads[src] * alpha.unsqueeze(-1); msg = msg.view(-1, self.out_dim * self.heads)
        out = torch.zeros(N, self.out_dim * self.heads, device=x.device, dtype=msg.dtype)
        out.scatter_add_(0, dst.unsqueeze(-1).expand_as(msg), msg)
        return self.act(out)

class ThreeD_GAT(nn.Module):
    def __init__(self, node_dim=11, edge_dim=23, hidden=HIDDEN, n_layers=N_LAYERS,
                 output_dim=8, dropout=DROPOUT, heads=GAT_HEADS):
        super().__init__(); self.heads = heads
        self.embed = nn.Linear(node_dim, hidden)
        self.convs = nn.ModuleList([GATConv(hidden, hidden//heads, edge_dim, heads=heads) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout)
        self.readout = nn.Sequential(
            nn.Linear(hidden*3, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden//2, output_dim))
    def forward(self, g):
        x, ei, ea = g['x'], g['edge_index'], g.get('edge_attr', None)
        h = self.embed(x)
        for conv, norm in zip(self.convs, self.norms): h = norm(h + conv(h, ei, ea)); h = self.drop(h)
        mm = (x[:,7] > 0.5)
        mf = h[mm].mean(dim=0) if mm.any() else torch.zeros(h.size(1), device=h.device, dtype=h.dtype)
        return self.readout(torch.cat([h.mean(0), h.max(0).values, mf])).unsqueeze(0)

model_3d = ThreeD_GAT().to(device)
if os.path.exists(CKPT_3D):
    state = torch.load(CKPT_3D, map_location=device, weights_only=False)
    # Strip torch.compile prefix if present
    clean_state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
    model_3d.load_state_dict(clean_state)
    print(f'3D GAT loaded from {CKPT_3D}')
else:
    print(f'WARNING: checkpoint not found ({CKPT_3D})')
model_3d.eval()
print(f'Params: {sum(p.numel() for p in model_3d.parameters()):,}')

# ===== SMILES -> 3D 图构建 (ETKDG + MMFF, RBF=16, BO=0) =====
import numpy as np
RBF_CENTERS = np.linspace(1.0, 5.0, 16)
RBF_SIGMA = 0.2

def smiles_to_3d_graph(smiles):
    """SMILES -> full 3D GAT graph (RBF distances, BO=0)"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    mol = Chem.AddHs(mol)
    try:
        if AllChem.EmbedMolecule(mol, randomSeed=42, maxAttempts=200) != 0:
            return None
        AllChem.MMFFOptimizeMolecule(mol)
    except:
        return None
    mol = Chem.RemoveHs(mol)
    conf = mol.GetConformer()
    N = mol.GetNumAtoms()

    # Node: SMILES 8d + BO stats 3d (all zero)
    xf = [atom_features(a) for a in mol.GetAtoms()]
    for i in range(N): xf[i].extend([0, 0.0, 0.0])
    x = torch.tensor(xf, dtype=torch.float32)

    # Edge: SMILES bonds 6d + RBF 16d + BO 1d
    src, dst, battr = [], [], []
    coords = np.array([[conf.GetAtomPosition(i).x,
                        conf.GetAtomPosition(i).y,
                        conf.GetAtomPosition(i).z] for i in range(N)])
    sm_z = [xf[i][0] for i in range(N)]

    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = int(bond.GetBondType())
        za, zb = sm_z[i], sm_z[j]
        is_ml = float((za>=21)!=(zb>=21))
        is_ring = float(bond.IsInRing())
        src.extend([i,j]); dst.extend([j,i])
        bf = [0.0]*4
        if bt in B2I: bf[B2I[bt]] = 1.0
        bf.extend([is_ring, is_ml])
        d = float(np.linalg.norm(coords[i]-coords[j]))
        for mu in RBF_CENTERS:
            bf.append(np.exp(-(d-mu)**2/RBF_SIGMA))
        bf.append(0.0)  # Mayer BO = 0
        battr.extend([bf,bf])

    ei = torch.tensor([src,dst], dtype=torch.long) if src else torch.zeros(2,0,dtype=torch.long)
    ea = torch.tensor(battr, dtype=torch.float32) if battr else torch.zeros(0,23)
    return {'x': x, 'edge_index': ei, 'edge_attr': ea}

print('3D GAT scorer ready (SMILES -> ETKDG+MMFF -> RBF -> 3D GAT -> 8 props)')


In [ ]:
# ============================================================
# Step 4: 13 个催化剂 + 3D GAT 打分
# ============================================================

# 13 in-house catalysts (5 Ni, 2 Pd, 4 Fe, 2 Zr)
candidates = {
    'Zr-cat2': 'Br[Zr](Br)([CH-]1C=CC=C1)[CH-]1C=CC=C1',
    'Zr-cat1': 'I[Zr](Br)([CH-]1C=CC=C1)[CH-]1C=CC=C1',
    'Ni-cat1': 'CO1CCO2CCO(C)[Ni]21(Br)Br',
    'Ni-cat2': 'Br[Ni]1([N]2=C(C3=[N]1C=CC(C(C)(C)C)=C3)C=C(C(C)(C)C)C=C2)Br',
    'Ni-cat3': 'Br[Ni]1([N]2=C3C4=[N]1C=CC(C5=CC=C(C(F)(F)F)C=C5)=C4C=CC3=C(C6=CC=C(C(F)(F)F)C=C6)C=C2)Br',
    'Ni-cat4': 'Br[Ni]1[N]2=C3C4=[N]1C=CC(C5=CC=C(C(F)(F)F)C=C5)=C4C=CC3=C(C6=CC=C(C(F)(F)F)C=C6)C=C2',
    'Ni-cat5': 'FC(C(C=C1)=CC=C1C2=C(C=C3)C4=[N]([Ni][N]5=C4C3=C(C6=CC=C(C(F)(F)F)C=C6)C=C5)C=C2)(F)F',
    'Pd-cat1': 'FC1(F)OC2=C(O1)[C@@]([C@@]3=C(OC(F)(F)O4)C4=CC=C3[P]([Pd][P](C5=CC=CC=C5)6C7=CC=CC=C7)(C8=CC=CC=C8)C9=CC=CC=C9)=C6C=C2',
    'Pd-cat2': 'CC1(C)C2=C(OC3=C4C=CC=C13)C([P]([Pd][P](C5=CC=CC=C5)4C6=CC=CC=C6)(C7=CC=CC=C7)C8=CC=CC=C8)=CC=C2',
    'Fe-cat1': '[CH-]1C=CC=C1.[CH-]1C=CC=C1.[Fe+2].P(c1ccccc1)(c2ccccc2)c3ccccc3.P(C1CCCCC1)C2CCCCC2',
    'Fe-cat2': 'O=[N+]1C(C)(C)CCCC1(C)C.Cl[Fe](Cl)Cl',
    'Fe-cat3': 'Cl[Fe](Cl)Cl.O=[N+]1C(C)(C)CCCC1C',
    'Fe-cat4': 'O=[N+]1C(C)(C)CCCC1(C)C.Cl[Fe](Cl)Cl',
}

targets = ['gap', 'homo', 'lumo', 'dipole', 'metal_charge', 'E_elec', 'E_disp', 'polarizability']
print(f'Scoring {len(candidates)} in-house catalysts (Ni/Pd/Fe/Zr) with 3D GAT...\n')

results = []
for name, smi in candidates.items():
    g = smiles_to_3d_graph(smi)
    if g is None:
        print(f'  SKIP {name}: conformer generation failed')
        continue
    g_gpu = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in g.items()}
    with torch.no_grad():
        pred = model_3d(g_gpu).cpu().numpy().flatten()

    results.append({
        'Name': name,
        'gap_Ha':        pred[0],
        'gap_eV':        pred[0] * 27.2114,
        'homo_Ha':       pred[1],
        'lumo_Ha':       pred[2],
        'dipole_D':      pred[3],
        'metal_q_e':     pred[4],
        'E_elec_Ha':     pred[5],
        'E_disp_Ha':     pred[6],
        'polariz_a03':   pred[7],
    })
    print(f'  {name}: gap={pred[0]:.4f} Ha ({pred[0]*27.2114:.2f} eV), dipole={pred[3]:.2f} D')

df = pd.DataFrame(results).sort_values('E_elec_Ha')
print(f'\n=== 3D GAT Scorer Results ({len(results)}/{len(candidates)} scored) ===')
cols_show = ['Name', 'gap_Ha', 'gap_eV', 'dipole_D', 'metal_q_e', 'E_elec_Ha']
print(df[cols_show].to_string(index=False))

df.to_csv('scorer_3d_results.csv', index=False)
print('\nSaved scorer_3d_results.csv')


In [ ]:
# 可视化排名
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
names = df['Name'].values
for ax, col, title, color in [
    (axes[0], 'E_elec_Ha', 'E_elec (lower=more stable)', 'steelblue'),
    (axes[1], 'gap_Ha', 'HOMO-LUMO Gap (Ha)', 'darkorange'),
    (axes[2], 'dipole_D', 'Dipole from 3D GAT (Debye)', 'forestgreen'),
]:
    ax.barh(range(len(names)), df[col].values, color=color)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel(col)
    ax.set_title(title)
    ax.invert_yaxis()
plt.suptitle('Candidate Scoring (3D GAT, single model)', fontsize=14)
plt.tight_layout(); plt.show()



## 输出说明

| 列 | 来源模型 | 说明 |
|----|---------|------|
| 全部 8 项性质 | **3D GAT** | 单一模型预测所有性质 |
| E_elec_Ha | 3D GAT | 按此项排序, 越低越稳定 |
| dipole_D | 3D GAT | 基于 ETKDG+MMFF 生成坐标的 RBF 距离编码 |
| gap_Ha / homo_Ha / lumo_Ha | 3D GAT | 前线轨道能量 |

> 输入仅需 SMILES 字符串。3D 坐标由 ETKDG 构象搜索 + MMFF 力场优化自动生成。
> Mayer bond orders 在推理时设为 0。
> 使用 12-head GAT x 384d x 5 layers checkpoint。
